# Fixed-Height Study - Notebook 1 of 3: Generate Trees and Sequences

## ▶ RUNTIME: **CPU only**
In Colab: **Runtime → Change runtime type → CPU**. Do not attach a GPU - this notebook does not use it, and an attached GPU still uses up your GPU quota while it sits idle.

This notebook simulates the data every method is evaluated on, and saves it to the team repo. Run it first. Notebooks 2 and 3 then read what it saves.

**The three notebooks (run in this order)**

| # | Notebook | Runtime | What it does |
|---|---|---|---|
| 1 | `1_Generate_Trees_and_Sequences` | **CPU** | Simulates the trees and the sequences and saves them to the team repo |
| 2 | `2_Run_Phyloformer` | **GPU** | Runs Phyloformer on the saved data and pushes the results |
| 3 | `3_Run_Other_Methods` | **CPU** | Runs FastME, FastTree and IQ-TREE on the same saved data and pushes the results |

Notebook 1 must finish (and push) first. Notebooks 2 and 3 are independent of each other, so they can run at the same time in separate sessions.

## What this study does

Every person gets **one clock rate** and runs the same grid of **5 tree sizes x 5 sequence lengths** at that rate, for both the **LG** and the **Cherry** substitution models.

| Person | Clock rate |
|---|---|
| Siqi | 0.1 |
| Deep | 0.2 |
| Mihir | 0.25 |
| Swetha | 0.4 |
| Darren | 0.5 |

**How the trees are made**
1. Random birth-death trees are simulated with `simulate_trees.py`.
2. Each tree is rescaled so its **height = 1** (height = the longest root-to-tip distance).
3. Every branch length is then multiplied by your **clock rate** (strict molecular clock).

So the root-to-tip distance of every tree equals your clock rate, i.e. the clock rate is the expected number of substitutions per site from root to tip.

**Sequences** are then simulated along these trees, once per model and sequence length. The **same trees** are used for LG and Cherry, so the two models are compared fairly. Cherry rounds odd lengths down (25 becomes 24); the real length is recorded later in the results as `seq_length_actual`.

**Where everything is saved** (all inside the team repo, under a new `fixed_height_all_methods` name, so earlier rounds are never touched)

```
simulated_data/fixed_height_all_methods/
├── trees/true/clock_rate=R/tree_size=N/                          true trees         (notebook 1)
├── sequences/clock_rate=R/model=M/tree_size=N/seq_length=L/      alignments         (notebook 1)
└── trees/inferred/clock_rate=R/model=M/tree_size=N/seq_length=L/method=X/   trees built by each method (notebooks 2 and 3)
results/fixed_height_all_methods/
├── treesize_seqlength_rateR_{lg|cherry}_phyloformer_results.csv   (notebook 2)
├── treesize_seqlength_rateR_{lg|cherry}_baselines_results.csv     (notebook 3)
└── failures_rateR_{generate|phyloformer|baselines}.csv            anything that failed, and why
```

**If the session disconnects:** run the notebook again from the top. Trees and sequences that are already saved are reused and skipped, so it carries on where it stopped. Data is pushed to the repo after every tree size.

## 1. Settings

In [ ]:
import os
BASE_DIR = globals().get("BASE_DIR", os.getcwd())   # folder that will hold Phyloformer/ and team_repo/ (stays fixed if cells are re-run)

# ---- Settings: change ONLY CLOCK_RATE (use your own rate; same value in all three notebooks) ----
CLOCK_RATE = 0.4          # Siqi 0.1, Deep 0.2, Mihir 0.25, Swetha 0.4, Darren 0.5
TREE_HEIGHT = 1.0         # every tree is rescaled to this height BEFORE the clock rate is applied
MODELS = ["LG", "Cherry"] # substitution models the sequences were simulated under
TREE_SIZES = [10, 25, 50, 75, 100]       # number of tips
SEQ_LENGTHS = [20, 25, 50, 100, 200]     # alignment length in amino acids
N_TREES = 20              # replicate trees per tree size

## 2. Setup
Installs what is needed and checks this is a CPU runtime. Takes a minute or two.

In [ ]:
# One-time setup. Prints a short checklist (details go to setup_log.txt).
# If it ends with "SETUP FAILED", fix the [FAIL] line and run this cell again.
import os
os.chdir(BASE_DIR)
!test -d team_repo || (git clone -q --filter=blob:none --sparse https://github.com/ACKarl/EPTI.git team_repo && git -C team_repo sparse-checkout set setup)
!python team_repo/setup/setup_pipeline.py --stage 1 --clock-rate {CLOCK_RATE}

In [ ]:
import os
import sys

PF_DIR = os.path.join(BASE_DIR, "Phyloformer")      # the tool being evaluated
TEAM_REPO = os.path.join(BASE_DIR, "team_repo")     # our shared repo (data and results are saved here)
sys.path.insert(0, os.path.join(TEAM_REPO, "setup"))
import fh_pipeline as fh                    # our helper functions (readable on GitHub: setup/fh_pipeline.py)

WORK_DIR = os.path.join(PF_DIR, "data", "fh_work")   # scratch space for temporary files (wiped for every grid cell)

os.chdir(PF_DIR)                            # the repo's own scripts expect to run from this folder

## 3. GitHub login
Asked once, so the run is not interrupted later.

In [ ]:
# Asked ONCE per session. The token is kept in memory only (never written to a file or shown).
# It is used to push after every tree size, so a disconnect never costs more than one tree size.
creds = fh.ask_github_login(TEAM_REPO)

## 4. Generate and save the data

For each tree size: make (or reuse) the true trees, check their heights, then simulate the sequences for every model and sequence length, and push. A grid cell that fails is written to `failures_rate..._generate.csv` and the run carries on.

In [ ]:
for size in TREE_SIZES:
    print(f"=== tree size {size} ===")
    try:
        # True trees: simulated, rescaled to TREE_HEIGHT, multiplied by CLOCK_RATE, height-checked, saved.
        # If a complete set is already saved, it is reused so sequences always match the saved trees.
        fh.get_or_make_true_trees(PF_DIR, TEAM_REPO, CLOCK_RATE, size, N_TREES, TREE_HEIGHT)
    except Exception as e:
        print(f"  !! could not make trees for tree size {size}: {e}")
        for seqlen in SEQ_LENGTHS:
            for model in MODELS:
                fh.save_failures(fh.results_dir(TEAM_REPO), CLOCK_RATE, "generate",
                                 {"model": model, "tree_size": size, "seq_length": seqlen},
                                 [{"method": "-", "tree_id": "ALL", "reason": "trees failed: " + str(e)[:150]}])
        continue

    for seqlen in SEQ_LENGTHS:
        for model in MODELS:
            print(f"  {model}, seq_length={seqlen}")
            cell = {"model": model, "tree_size": size, "seq_length": seqlen}
            try:
                # Simulates the alignments for this model and length and saves them.
                # Returns the ids of any trees that did not get an alignment.
                missing = fh.generate_sequences(PF_DIR, TEAM_REPO, CLOCK_RATE, model, size, seqlen, N_TREES)
                rows = [{"method": "-", "tree_id": m, "reason": "alignment not produced"} for m in missing]
            except Exception as e:
                rows = [{"method": "-", "tree_id": "ALL", "reason": str(e)[:200]}]
            fh.save_failures(fh.results_dir(TEAM_REPO), CLOCK_RATE, "generate", cell, rows)

    # Push after every tree size. If the push fails the data stays here and the last cell pushes it again.
    try:
        fh.push_to_github(TEAM_REPO, fh.study_prefixes(CLOCK_RATE),
                          f"Fixed-height all-methods data: clock rate {CLOCK_RATE}, tree size {size}", creds)
    except Exception as e:
        print(f"  !! PUSH FAILED (data is safe locally, the final cell will try again): {str(e)[:200]}")

print("Finished.")

## 5. Check the data is complete
Counts the saved files against what was expected.

In [ ]:
problems = []
for size in TREE_SIZES:
    n = len(fh.list_files(fh.true_trees_dir(TEAM_REPO, CLOCK_RATE, size), fh.TREE_EXTS))
    if n != N_TREES:
        problems.append(f"true trees, tree_size={size}: {n}/{N_TREES}")
    for model in MODELS:
        for seqlen in SEQ_LENGTHS:
            n = len(fh.list_files(fh.sequences_dir(TEAM_REPO, CLOCK_RATE, model, size, seqlen), fh.ALN_EXTS))
            if n != N_TREES:
                problems.append(f"{model} sequences, tree_size={size}, seq_length={seqlen}: {n}/{N_TREES}")
print("All trees and sequences are complete." if not problems else "Incomplete:\n  " + "\n  ".join(problems))

## 6. Final push
Pushes anything that is not in the repo yet (safe to run even if everything is already pushed). Notebooks 2 and 3 can start once this says done.

In [ ]:
fh.push_to_github(TEAM_REPO, fh.study_prefixes(CLOCK_RATE),
                  f"Fixed-height all-methods data: clock rate {CLOCK_RATE} (final)", creds)